In [1]:
! pip install mlflow


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import numpy as np
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

import mlflow
import mlflow.sklearn
import mlflow.xgboost

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score)

# Set MLflow tracking URI — saves experiments locally
mlflow.set_tracking_uri('file:../mlruns')

print(f"MLflow version  : {mlflow.__version__}")
print(f"Tracking URI    : {mlflow.get_tracking_uri()}")

MLflow version  : 3.10.1
Tracking URI    : file:../mlruns


In [ ]:
X_train = pd.read_csv('../data/processed/X_train_eng.csv')
X_val   = pd.read_csv('../data/processed/X_val_eng.csv')
X_test  = pd.read_csv('../data/processed/X_test_eng.csv')

yc_train = pd.read_csv('../data/processed/yc_train.csv').squeeze()
yc_val   = pd.read_csv('../data/processed/yc_val.csv').squeeze()
yc_test  = pd.read_csv('../data/processed/yc_test.csv').squeeze()

yr_train = pd.read_csv('../data/processed/yr_train.csv').squeeze()
yr_val   = pd.read_csv('../data/processed/yr_val.csv').squeeze()
yr_test  = pd.read_csv('../data/processed/yr_test.csv').squeeze()

scaler = joblib.load('../models/scaler.pkl')

X_train_scaled = scaler.transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

with open('../data/processed/feature_cols_engineered.json') as f:
    feature_cols = json.load(f)

print(f"Data loaded successfully")
print(f"Train : {X_train.shape} | Val : {X_val.shape} | Test : {X_test.shape}")

Data loaded successfully
Train : (283521, 38) | Val : (60559, 38) | Test : (60720, 38)


In [7]:
# Create two separate experiments — one for each problem
clf_experiment = mlflow.set_experiment('EMI_Classification')
reg_experiment = mlflow.set_experiment('EMI_Regression')

print(f"Classification Experiment ID : {clf_experiment.experiment_id}")
print(f"Regression Experiment ID     : {reg_experiment.experiment_id}")

Classification Experiment ID : 726149996539529412
Regression Experiment ID     : 647582438171220886


In [5]:
mlflow.set_experiment('EMI_Classification')

classification_models = {
    'Logistic_Regression': (
        LogisticRegression(class_weight='balanced', max_iter=1000,
                           random_state=42, n_jobs=-1),
        X_train_scaled, X_val_scaled, X_test_scaled
    ),
    'Decision_Tree': (
        DecisionTreeClassifier(max_depth=10, min_samples_split=50,
                               min_samples_leaf=20, class_weight='balanced',
                               random_state=42),
        X_train, X_val, X_test
    ),
    'Random_Forest': (
        RandomForestClassifier(n_estimators=100, max_depth=15,
                               min_samples_split=50, min_samples_leaf=20,
                               class_weight='balanced', random_state=42,
                               n_jobs=-1),
        X_train, X_val, X_test
    ),
    'XGBoost': (
        XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                      subsample=0.8, colsample_bytree=0.8,
                      eval_metric='mlogloss', random_state=42, n_jobs=-1),
        X_train, X_val, X_test
    )
}

for model_name, (model, X_tr, X_v, X_te) in classification_models.items():
    print(f"\nLogging {model_name}...")
    with mlflow.start_run(run_name=model_name):

        # Train
        if model_name == 'XGBoost':
            sw = compute_sample_weight('balanced', yc_train)
            model.fit(X_tr, yc_train, sample_weight=sw)
        else:
            model.fit(X_tr, yc_train)

        # Predictions
        y_pred = model.predict(X_v)

        # Metrics
        acc  = accuracy_score(yc_val, y_pred)
        prec = precision_score(yc_val, y_pred, average='weighted')
        rec  = recall_score(yc_val, y_pred, average='weighted')
        f1   = f1_score(yc_val, y_pred, average='weighted')
        auc  = roc_auc_score(yc_val, model.predict_proba(X_v),
                             multi_class='ovr', average='weighted')

        # Log parameters
        mlflow.log_params(model.get_params())

        # Log metrics
        mlflow.log_metrics({
            'val_accuracy'  : round(acc, 4),
            'val_precision' : round(prec, 4),
            'val_recall'    : round(rec, 4),
            'val_f1_score'  : round(f1, 4),
            'val_roc_auc'   : round(auc, 4)
        })

        # Log model
        if 'XGBoost' in model_name:
            mlflow.xgboost.log_model(model, artifact_path=model_name)
        else:
            mlflow.sklearn.log_model(model, artifact_path=model_name)

        print(f"  Accuracy : {acc*100:.2f}%  F1 : {f1*100:.2f}%  AUC : {auc*100:.2f}%")

print("\n✅ All classification models logged to MLflow")


Logging Logistic_Regression...


2026/05/23 17:01:06 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/05/23 17:01:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instea

  Accuracy : 80.19%  F1 : 85.09%  AUC : 96.47%

Logging Decision_Tree...


2026/05/23 17:01:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/23 17:01:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  Accuracy : 88.81%  F1 : 91.32%  AUC : 98.46%

Logging Random_Forest...


2026/05/23 17:03:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/23 17:03:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  Accuracy : 90.03%  F1 : 92.04%  AUC : 99.21%

Logging XGBoost...


2026/05/23 17:04:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Accuracy : 92.29%  F1 : 93.80%  AUC : 99.65%

✅ All classification models logged to MLflow


In [7]:
# log regression models
mlflow.set_experiment('EMI_Regression')

regression_models = {
    'Linear_Regression': (
        LinearRegression(n_jobs=-1),
        X_train_scaled, X_val_scaled, X_test_scaled
    ),
    'Decision_Tree': (
        DecisionTreeRegressor(max_depth=10, min_samples_split=50,
                              min_samples_leaf=20, random_state=42),
        X_train, X_val, X_test
    ),
    'Random_Forest': (
        RandomForestRegressor(n_estimators=100, max_depth=15,
                              min_samples_split=50, min_samples_leaf=20,
                              random_state=42, n_jobs=-1),
        X_train, X_val, X_test
    ),
    'XGBoost': (
        XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1,
                     subsample=0.8, colsample_bytree=0.8,
                     random_state=42, n_jobs=-1),
        X_train, X_val, X_test
    )
}

for model_name, (model, X_tr, X_v, X_te) in regression_models.items():
    print(f"\nLogging {model_name}...")
    with mlflow.start_run(run_name=model_name):

        # Train
        model.fit(X_tr, yr_train)

        # Predictions
        y_pred = model.predict(X_v)

        # Metrics
        rmse = np.sqrt(mean_squared_error(yr_val, y_pred))
        mae  = mean_absolute_error(yr_val, y_pred)
        r2   = r2_score(yr_val, y_pred)
        mape = np.mean(np.abs((yr_val - y_pred) / (yr_val + 1))) * 100

        # Log parameters
        mlflow.log_params(model.get_params())

        # Log metrics
        mlflow.log_metrics({
            'val_rmse'  : round(rmse, 2),
            'val_mae'   : round(mae, 2),
            'val_r2'    : round(r2, 4),
            'val_mape'  : round(mape, 2)
        })

        # Log model
        if 'XGBoost' in model_name:
            mlflow.xgboost.log_model(model, artifact_path=model_name)
        else:
            mlflow.sklearn.log_model(model, artifact_path=model_name)

        print(f"  RMSE : {rmse:,.2f}  MAE : {mae:,.2f}  R2 : {r2:.4f}")

print("\n✅ All regression models logged to MLflow")


Logging Linear_Regression...


2026/05/23 17:24:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/23 17:24:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  RMSE : 3,981.37  MAE : 2,826.28  R2 : 0.7301

Logging Decision_Tree...


2026/05/23 17:25:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/23 17:25:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  RMSE : 1,300.07  MAE : 538.49  R2 : 0.9712

Logging Random_Forest...


2026/05/23 17:36:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/23 17:36:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  RMSE : 1,058.75  MAE : 292.64  R2 : 0.9809

Logging XGBoost...


2026/05/23 17:37:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  RMSE : 807.94  MAE : 310.81  R2 : 0.9889

✅ All regression models logged to MLflow


In [8]:

# Register best classification model — XGBoost
print("Registering best models in MLflow registry...")

# Get the XGBoost run ID from classification experiment
clf_runs = mlflow.search_runs(
    experiment_names=['EMI_Classification'],
    order_by=['metrics.val_accuracy DESC']
)

best_clf_run_id = clf_runs.iloc[0]['run_id']
best_clf_name   = clf_runs.iloc[0]['tags.mlflow.runName']
print(f"Best classifier : {best_clf_name} | Run ID : {best_clf_run_id}")

# Get the XGBoost run ID from regression experiment
reg_runs = mlflow.search_runs(
    experiment_names=['EMI_Regression'],
    order_by=['metrics.val_r2 DESC']
)

best_reg_run_id = reg_runs.iloc[0]['run_id']
best_reg_name   = reg_runs.iloc[0]['tags.mlflow.runName']
print(f"Best regressor  : {best_reg_name} | Run ID : {best_reg_run_id}")

Registering best models in MLflow registry...
Best classifier : XGBoost | Run ID : 8a60521782414f07b4bea705f2656f26
Best regressor  : XGBoost | Run ID : 7fc28f7590b949ff905f52becadb622d


In [9]:
# Register models
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Register best classifier
clf_model_uri = f"runs:/{best_clf_run_id}/{best_clf_name}"
clf_registered = mlflow.register_model(
    model_uri=clf_model_uri,
    name='EMI_Best_Classifier'
)
print(f"✅ Classifier registered : {clf_registered.name} v{clf_registered.version}")

# Register best regressor
reg_model_uri = f"runs:/{best_reg_run_id}/{best_reg_name}"
reg_registered = mlflow.register_model(
    model_uri=reg_model_uri,
    name='EMI_Best_Regressor'
)
print(f"✅ Regressor registered  : {reg_registered.name} v{reg_registered.version}")

Successfully registered model 'EMI_Best_Classifier'.
2026/05/23 17:38:00 WARNING mlflow.tracking._model_registry.fluent: Run with id 8a60521782414f07b4bea705f2656f26 has no artifacts at artifact path 'XGBoost', registering model based on models:/m-fd531b1ab2184572b02092cd353d80a2 instead
Created version '1' of model 'EMI_Best_Classifier'.


✅ Classifier registered : EMI_Best_Classifier v1


Successfully registered model 'EMI_Best_Regressor'.
2026/05/23 17:38:01 WARNING mlflow.tracking._model_registry.fluent: Run with id 7fc28f7590b949ff905f52becadb622d has no artifacts at artifact path 'XGBoost', registering model based on models:/m-584a2ca789cd4bb08f75661bdd960101 instead
Created version '1' of model 'EMI_Best_Regressor'.


✅ Regressor registered  : EMI_Best_Regressor v1


In [1]:
import mlflow
print(mlflow.get_tracking_uri())

file:///c:/Users/HP/Downloads/emi_prediction/mlruns


In [2]:
import os
# Check if mlruns folder exists and where
for root, dirs, files in os.walk('..'):
    for d in dirs:
        if d == 'mlruns':
            print(os.path.join(root, d))

..\mlruns
..\emi_prediction\mlruns


In [8]:
import mlflow

# Set tracking URI to exact absolute path
mlflow.set_tracking_uri(r"file:///C:/Users/HP/Downloads/emi_prediction/mlruns")

# Verify
print(f"Tracking URI : {mlflow.get_tracking_uri()}")

# Create experiments
clf_experiment = mlflow.set_experiment('EMI_Classification')
reg_experiment = mlflow.set_experiment('EMI_Regression')

print(f"Classification Experiment ID : {clf_experiment.experiment_id}")
print(f"Regression Experiment ID     : {reg_experiment.experiment_id}")

Tracking URI : file:///C:/Users/HP/Downloads/emi_prediction/mlruns
Classification Experiment ID : 525522749045219809
Regression Experiment ID     : 850641607816962418


In [9]:
mlflow.set_experiment('EMI_Classification')

classification_models = {
    'Logistic_Regression': (
        LogisticRegression(class_weight='balanced', max_iter=1000,
                           random_state=42, n_jobs=-1),
        X_train_scaled, X_val_scaled, X_test_scaled
    ),
    'Decision_Tree': (
        DecisionTreeClassifier(max_depth=10, min_samples_split=50,
                               min_samples_leaf=20, class_weight='balanced',
                               random_state=42),
        X_train, X_val, X_test
    ),
    'Random_Forest': (
        RandomForestClassifier(n_estimators=100, max_depth=15,
                               min_samples_split=50, min_samples_leaf=20,
                               class_weight='balanced', random_state=42,
                               n_jobs=-1),
        X_train, X_val, X_test
    ),
    'XGBoost': (
        XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                      subsample=0.8, colsample_bytree=0.8,
                      eval_metric='mlogloss', random_state=42, n_jobs=-1),
        X_train, X_val, X_test
    )
}

for model_name, (model, X_tr, X_v, X_te) in classification_models.items():
    print(f"Logging {model_name}...")
    with mlflow.start_run(run_name=model_name):
        if model_name == 'XGBoost':
            sw = compute_sample_weight('balanced', yc_train)
            model.fit(X_tr, yc_train, sample_weight=sw)
        else:
            model.fit(X_tr, yc_train)

        y_pred = model.predict(X_v)
        acc  = accuracy_score(yc_val, y_pred)
        prec = precision_score(yc_val, y_pred, average='weighted')
        rec  = recall_score(yc_val, y_pred, average='weighted')
        f1   = f1_score(yc_val, y_pred, average='weighted')
        auc  = roc_auc_score(yc_val, model.predict_proba(X_v),
                             multi_class='ovr', average='weighted')

        mlflow.log_params(model.get_params())
        mlflow.log_metrics({
            'val_accuracy'  : round(acc, 4),
            'val_precision' : round(prec, 4),
            'val_recall'    : round(rec, 4),
            'val_f1_score'  : round(f1, 4),
            'val_roc_auc'   : round(auc, 4)
        })

        if 'XGBoost' in model_name:
            mlflow.xgboost.log_model(model, artifact_path=model_name)
        else:
            mlflow.sklearn.log_model(model, artifact_path=model_name)

        print(f"  ✅ Accuracy: {acc*100:.2f}%  F1: {f1*100:.2f}%  AUC: {auc*100:.2f}%")

print("\n✅ All classification models logged")

2026/05/23 18:08:16 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Logging Logistic_Regression...


2026/05/23 18:08:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/23 18:08:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  ✅ Accuracy: 80.19%  F1: 85.09%  AUC: 96.47%
Logging Decision_Tree...


2026/05/23 18:08:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/23 18:08:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  ✅ Accuracy: 88.81%  F1: 91.32%  AUC: 98.46%
Logging Random_Forest...


2026/05/23 18:09:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/23 18:09:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  ✅ Accuracy: 90.03%  F1: 92.04%  AUC: 99.21%
Logging XGBoost...


2026/05/23 18:09:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  ✅ Accuracy: 92.29%  F1: 93.80%  AUC: 99.65%

✅ All classification models logged


In [10]:
mlflow.set_experiment('EMI_Regression')

regression_models = {
    'Linear_Regression': (
        LinearRegression(n_jobs=-1),
        X_train_scaled, X_val_scaled, X_test_scaled
    ),
    'Decision_Tree': (
        DecisionTreeRegressor(max_depth=10, min_samples_split=50,
                              min_samples_leaf=20, random_state=42),
        X_train, X_val, X_test
    ),
    'Random_Forest': (
        RandomForestRegressor(n_estimators=100, max_depth=15,
                              min_samples_split=50, min_samples_leaf=20,
                              random_state=42, n_jobs=-1),
        X_train, X_val, X_test
    ),
    'XGBoost': (
        XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1,
                     subsample=0.8, colsample_bytree=0.8,
                     random_state=42, n_jobs=-1),
        X_train, X_val, X_test
    )
}

for model_name, (model, X_tr, X_v, X_te) in regression_models.items():
    print(f"Logging {model_name}...")
    with mlflow.start_run(run_name=model_name):
        model.fit(X_tr, yr_train)
        y_pred = model.predict(X_v)

        rmse = np.sqrt(mean_squared_error(yr_val, y_pred))
        mae  = mean_absolute_error(yr_val, y_pred)
        r2   = r2_score(yr_val, y_pred)
        mape = np.mean(np.abs((yr_val - y_pred) / (yr_val + 1))) * 100

        mlflow.log_params(model.get_params())
        mlflow.log_metrics({
            'val_rmse' : round(rmse, 2),
            'val_mae'  : round(mae, 2),
            'val_r2'   : round(r2, 4),
            'val_mape' : round(mape, 2)
        })

        if 'XGBoost' in model_name:
            mlflow.xgboost.log_model(model, artifact_path=model_name)
        else:
            mlflow.sklearn.log_model(model, artifact_path=model_name)

        print(f"  ✅ RMSE: {rmse:,.2f}  MAE: {mae:,.2f}  R2: {r2:.4f}")

print("\n✅ All regression models logged")

Logging Linear_Regression...


2026/05/23 18:11:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/23 18:11:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  ✅ RMSE: 3,981.37  MAE: 2,826.28  R2: 0.7301
Logging Decision_Tree...


2026/05/23 18:11:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/23 18:11:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  ✅ RMSE: 1,300.07  MAE: 538.49  R2: 0.9712
Logging Random_Forest...


2026/05/23 18:14:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/23 18:14:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  ✅ RMSE: 1,058.75  MAE: 292.64  R2: 0.9809
Logging XGBoost...


2026/05/23 18:15:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  ✅ RMSE: 807.94  MAE: 310.81  R2: 0.9889

✅ All regression models logged


In [11]:
pip install streamlit

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
for root, dirs, files in os.walk('.'):
    for file in files:
        if file.endswith('.pkl'):
            print(os.path.join(root, file))

.\columns.pkl
.\target_encoder.pkl
.\mlruns\525522749045219809\models\m-4284bf830dca4362ad1d207bc930658d\artifacts\model.pkl
.\mlruns\525522749045219809\models\m-b737b7faa96a4390b9378311e023ba52\artifacts\model.pkl
.\mlruns\525522749045219809\models\m-e347ccb81eac49e5ab55772dfee64761\artifacts\model.pkl
.\mlruns\850641607816962418\models\m-1b54fbc132af442f885bd945bbbeea68\artifacts\model.pkl
.\mlruns\850641607816962418\models\m-316dfb7395694dd09c069ff2025453c5\artifacts\model.pkl
.\mlruns\850641607816962418\models\m-688c6101e2074329b6a379e5a594578e\artifacts\model.pkl


In [2]:
import joblib
import os
import pandas as pd
import json
from xgboost import XGBClassifier, XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

os.makedirs('models', exist_ok=True)

# Load data
X_train = pd.read_csv('data/processed/X_train_eng.csv')
X_val   = pd.read_csv('data/processed/X_val_eng.csv')
X_test  = pd.read_csv('data/processed/X_test_eng.csv')

yc_train = pd.read_csv('data/processed/yc_train.csv').squeeze()
yr_train = pd.read_csv('data/processed/yr_train.csv').squeeze()

# Scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Train best classifier
print("Training classifier...")
sw = compute_sample_weight('balanced', yc_train)
clf = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                    subsample=0.8, colsample_bytree=0.8,
                    eval_metric='mlogloss', random_state=42, n_jobs=-1)
clf.fit(X_train, yc_train, sample_weight=sw)
joblib.dump(clf, 'models/best_classifier.pkl')
print("✅ best_classifier.pkl saved")

# Train best regressor
print("Training regressor...")
reg = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1,
                   subsample=0.8, colsample_bytree=0.8,
                   random_state=42, n_jobs=-1)
reg.fit(X_train, yr_train)
joblib.dump(reg, 'models/best_regressor.pkl')
print("✅ best_regressor.pkl saved")

# Save scaler
joblib.dump(scaler, 'models/scaler.pkl')
print("✅ scaler.pkl saved")

print("\nAll models saved to models/ folder:")
print(os.listdir('models'))

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/X_train_eng.csv'

In [3]:
import os
print(os.getcwd())

c:\Users\HP\Downloads\emi_prediction


In [6]:
for root, dirs, files in os.walk('.'):
    for file in files:
        if 'X_train_eng' in file:
            print(os.path.join(root, file))

In [7]:
import os

# Check if data folder exists
print("Folders in current directory:")
print(os.listdir('.'))

Folders in current directory:
['app.py', 'columns.pkl', 'EDA.ipynb', 'emi_prediction_dataset.csv', 'Feature_engineering.ipynb', 'mlflow.db', 'mlflow.ipynb', 'mlruns', 'models', 'models.ipynb', 'pages', 'process.ipynb', 'Regression_models.ipynb', 'requirements.txt', 'target_encoder.pkl', 'utils.py']


In [8]:
import os

# Search for X_train_eng.csv anywhere
for root, dirs, files in os.walk('C:\\Users\\HP\\Downloads'):
    for file in files:
        if 'X_train_eng' in file or 'df_engineered' in file:
            print(os.path.join(root, file))

C:\Users\HP\Downloads\data\processed\df_engineered.csv
C:\Users\HP\Downloads\data\processed\X_train_eng.csv


In [9]:
import shutil
import os

# Copy data folder into project
src = r'C:\Users\HP\Downloads\data'
dst = r'C:\Users\HP\Downloads\emi_prediction\data'

shutil.copytree(src, dst)
print("✅ Data folder copied successfully")
print(os.listdir(r'C:\Users\HP\Downloads\emi_prediction\data\processed'))

✅ Data folder copied successfully
['age_by_eligibility.png', 'avg_emi_by_scenario.png', 'boxplots_by_eligibility.png', 'classification_comparison.png', 'correlation_heatmap.png', 'creditscore_by_eligibility.png', 'df_clean.csv', 'df_engineered.csv', 'eligibility_by_employment.png', 'eligibility_by_scenario.png', 'emi_by_education.png', 'engineered_feature_correlations.png', 'feature_cols.json', 'feature_cols_engineered.json', 'regression_comparison.png', 'salary_by_eligibility.png', 'scenario_distribution.png', 'target_distribution.png', 'top_feature_correlations.png', 'xgb_confusion_matrix.png', 'xgb_feature_importance.png', 'xgb_regression_plots.png', 'X_test.csv', 'X_test_eng.csv', 'X_train.csv', 'X_train_eng.csv', 'X_val.csv', 'X_val_eng.csv', 'yc_test.csv', 'yc_train.csv', 'yc_val.csv', 'yr_test.csv', 'yr_train.csv', 'yr_val.csv']


In [10]:
import joblib
import os
import pandas as pd
from xgboost import XGBClassifier, XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

os.makedirs('models', exist_ok=True)

# Load data with correct path
X_train  = pd.read_csv('data/processed/X_train_eng.csv')
yc_train = pd.read_csv('data/processed/yc_train.csv').squeeze()
yr_train = pd.read_csv('data/processed/yr_train.csv').squeeze()

print(f"Data loaded: {X_train.shape}")

# Scaler
scaler = StandardScaler()
scaler.fit(X_train)
joblib.dump(scaler, 'models/scaler.pkl')
print("✅ scaler.pkl saved")

# Best Classifier — XGBoost
print("Training classifier...")
sw  = compute_sample_weight('balanced', yc_train)
clf = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                    subsample=0.8, colsample_bytree=0.8,
                    eval_metric='mlogloss', random_state=42, n_jobs=-1)
clf.fit(X_train, yc_train, sample_weight=sw)
joblib.dump(clf, 'models/best_classifier.pkl')
print("✅ best_classifier.pkl saved")

# Best Regressor — XGBoost
print("Training regressor...")
reg = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1,
                   subsample=0.8, colsample_bytree=0.8,
                   random_state=42, n_jobs=-1)
reg.fit(X_train, yr_train)
joblib.dump(reg, 'models/best_regressor.pkl')
print("✅ best_regressor.pkl saved")

print("\nModels saved:")
print(os.listdir('models'))

Data loaded: (283521, 38)
✅ scaler.pkl saved
Training classifier...
✅ best_classifier.pkl saved
Training regressor...
✅ best_regressor.pkl saved

Models saved:
['best_classifier.pkl', 'best_regressor.pkl', 'scaler.pkl']
